In [15]:
import os, sys
notebook_dir = os.path.dirname(os.path.abspath("__file__")) if '__file__' in globals() else os.getcwd()
cds_root = os.path.abspath(os.path.join(notebook_dir, ".."))
if cds_root not in sys.path:
    sys.path.insert(0, cds_root)
src_path = os.path.join(cds_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

In [ ]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset as TorchDataset
import torchvision
from torchvision import transforms
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Dinov2Model,
    Wav2Vec2Processor,
    Wav2Vec2Model,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from datasets import Dataset
from src.data.image.data_loader import ImageDataset
import librosa

# =====================
# Device
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================
# Paths
# =====================
test_metadata_csv = r"C:\Users\User\Downloads\cds\CDS\processed_test_metadata.csv"
checkpoint_path = r"C:\Users\User\Downloads\cds\CDS\text_checkpoint\checkpoint-2500"
image_checkpoint_path = r"C:\Users\User\Downloads\cds\CDS\best_dinov2_emotion.pt"
audio_checkpoint_path = r"C:\Users\User\Downloads\cds\CDS\checkpoint.pt"
frames_root_dir = r"C:\Users\User\Downloads\cds\CDS\processed_test_frames\\"
audio_dir = r"C:\Users\User\Downloads\cds\CDS\processed_test_audio"


In [17]:
# =====================
# Load test metadata
# =====================
test_df = pd.read_csv(test_metadata_csv)[["video_id", "utterance", "emotion"]]
label_list = sorted(test_df["emotion"].unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
test_df["label"] = test_df["emotion"].map(label2id)

# Build video_id 
def clean_utterance(text):
    import re
    from unidecode import unidecode
    text = str(text)
    text = text.replace('…', '...').replace('\xa0', ' ')
    text = unidecode(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

video_to_text = {
    row["video_id"]: clean_utterance(row["utterance"])
    for _, row in test_df.iterrows()
}

# DINOv2 transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

In [18]:
# =====================
# Text Model Loading
# =====================
tokenizer = AutoTokenizer.from_pretrained("bhadresh-savani/bert-base-uncased-emotion")
text_model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
text_model = text_model.to(device)
text_model.eval()

# =====================
# Image Model Loading
# =====================
class DINOv2EmotionClassifier(nn.Module):
    def __init__(self, num_labels, model_name="facebook/dinov2-base"):
        super().__init__()
        self.backbone = Dinov2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, num_labels),
        )
        self.freeze_backbone()

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values=pixel_values).last_hidden_state
        cls_token = outputs[:, 0, :]
        return self.classifier(cls_token)

image_model = DINOv2EmotionClassifier(num_labels=len(label_list)).to(device)
image_ckpt = torch.load(image_checkpoint_path, map_location=device, weights_only=False)
image_model.load_state_dict(image_ckpt, strict=False)
image_model.eval()

# =====================
# Audio Model Loading
# =====================
class Wav2Vec2EmotionClassifier(nn.Module):
    def __init__(self, num_classes, model_name="facebook/wav2vec2-base"):
        super().__init__()
        self.wav2vec2 = Wav2Vec2Model.from_pretrained(model_name)
        self.freeze_encoder()
        hidden_size = self.wav2vec2.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def freeze_encoder(self):
        for p in self.wav2vec2.parameters():
            p.requires_grad = False

    def forward(self, input_values):
        hidden = self.wav2vec2(input_values).last_hidden_state
        pooled = hidden.mean(dim=1)
        return self.classifier(pooled)

audio_ckpt = torch.load(audio_checkpoint_path, map_location=device, weights_only=False)
audio_label_list = [str(x) for x in audio_ckpt["label_classes"]]
audio_label2id = {label: idx for idx, label in enumerate(audio_label_list)}
processor = Wav2Vec2Processor.from_pretrained(audio_ckpt["model_name"])

audio_model = Wav2Vec2EmotionClassifier(
    num_classes=len(audio_label_list),
    model_name=audio_ckpt["model_name"],
).to(device)
audio_model.load_state_dict(audio_ckpt["model_state_dict"])
audio_model.eval()


Loading weights: 100%|██████████| 211/211 [00:00<00:00, 22684.70it/s]
Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_q.weight             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Wav2Vec2EmotionClassifier(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  

In [19]:
# =====================
# Prepare Aligned Datasets
# =====================
sys.path.append(r"C:\Users\User\Downloads\cds\CDS")
from src.utils import load_video_data

test_video_dirs, test_labels, test_emotion_to_idx = load_video_data(
    metadata_csv=test_metadata_csv,
    frames_root_dir=frames_root_dir
)

texts_aligned = []
aligned_labels = []
aligned_video_dirs = []

for video_dir, label in zip(test_video_dirs, test_labels):
    video_id = os.path.basename(video_dir)
    if video_id not in video_to_text:
        continue
    if not os.path.exists(video_dir) or len(os.listdir(video_dir)) == 0:
        continue
    texts_aligned.append(video_to_text[video_id])
    aligned_labels.append(label)
    aligned_video_dirs.append(video_dir)

# Image dataset
image_dataset = ImageDataset(
    video_dirs=aligned_video_dirs,
    labels=aligned_labels,
    transform=transform,
    n_frames=8
)
image_loader = DataLoader(image_dataset, batch_size=8, shuffle=False, num_workers=0)

# MELD label -> audio model label
fusion_name_map = {
    "anger":    "angry",
    "fear":     "fearful",
    "joy":      "happy",
    "sadness":  "sad",
    "surprise": "surprised",
}

# Audio dataset
class RawAudioDataset(TorchDataset):
    def __init__(self, df, audio_dir, label2id, name_map=None, sample_rate=16000):
        self.video_ids   = df["video_id"].tolist()
        self.emotions    = df["emotion"].tolist()
        self.audio_dir   = audio_dir
        self.label2id    = label2id
        self.name_map    = name_map or {}
        self.sample_rate = sample_rate

    def __len__(self):
        return len(self.video_ids)

    def __getitem__(self, idx):
        audio_path = os.path.join(self.audio_dir, self.video_ids[idx] + ".wav")
        waveform, sr = librosa.load(audio_path, sr=self.sample_rate, mono=True)
        if len(waveform) == 0:
            waveform = np.zeros(self.sample_rate, dtype=np.float32)
        emotion = self.name_map.get(self.emotions[idx], self.emotions[idx])
        return waveform.astype(np.float32), self.label2id[emotion]

aligned_video_ids = [os.path.basename(d) for d in aligned_video_dirs]
id_order = {vid: i for i, vid in enumerate(aligned_video_ids)}

test_df_audio = pd.read_csv(test_metadata_csv)[["video_id", "emotion"]]
test_df_audio = test_df_audio[test_df_audio["video_id"].isin(aligned_video_ids)]
test_df_audio = test_df_audio.sort_values(
    "video_id", key=lambda col: col.map(id_order)
).reset_index(drop=True)

audio_to_fusion_idx = {
    audio_label2id[fusion_name_map.get(label, label)]: fusion_idx
    for label, fusion_idx in label2id.items()
    if fusion_name_map.get(label, label) in audio_label2id
}

class AudioCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, batch):
        waveforms, labels = zip(*batch)
        inputs = self.processor(
            list(waveforms),
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )
        return inputs.input_values.to(device), torch.LongTensor(labels).to(device)

audio_dataset = RawAudioDataset(test_df_audio, audio_dir, audio_label2id, name_map=fusion_name_map)
audio_loader = DataLoader(
    audio_dataset, batch_size=8, shuffle=False,
    collate_fn=AudioCollator(processor), num_workers=0,
)


In [20]:
# =====================
# Prepare Train Aligned Datasets
# =====================
train_metadata_csv  = r"C:\Users\User\Downloads\cds\CDS\processed_train_metadata.csv"
train_frames_root   = r"C:\Users\User\Downloads\cds\CDS\processed_train_frames\\"
train_audio_root    = r"C:\Users\User\Downloads\cds\CDS\processed_train_audio"

train_df_meta = pd.read_csv(train_metadata_csv)[["video_id", "utterance", "emotion"]]
train_video_to_text = {
    row["video_id"]: clean_utterance(row["utterance"])
    for _, row in train_df_meta.iterrows()
}

train_video_dirs_all, train_labels_all, _ = load_video_data(
    metadata_csv=train_metadata_csv,
    frames_root_dir=train_frames_root
)

train_texts_aligned    = []
train_aligned_labels   = []
train_aligned_video_dirs = []

for video_dir, label in zip(train_video_dirs_all, train_labels_all):
    video_id = os.path.basename(video_dir)
    if video_id not in train_video_to_text:
        continue
    if not os.path.exists(video_dir) or len(os.listdir(video_dir)) == 0:
        continue
    train_texts_aligned.append(train_video_to_text[video_id])
    train_aligned_labels.append(label)
    train_aligned_video_dirs.append(video_dir)

train_image_dataset = ImageDataset(
    video_dirs=train_aligned_video_dirs,
    labels=train_aligned_labels,
    transform=transform,
    n_frames=8
)
train_image_loader = DataLoader(train_image_dataset, batch_size=8, shuffle=False, num_workers=0)

train_aligned_video_ids = [os.path.basename(d) for d in train_aligned_video_dirs]
train_id_order = {vid: i for i, vid in enumerate(train_aligned_video_ids)}

train_df_audio = pd.read_csv(train_metadata_csv)[["video_id", "emotion"]]
train_df_audio = train_df_audio[train_df_audio["video_id"].isin(train_aligned_video_ids)]
train_df_audio = train_df_audio.sort_values(
    "video_id", key=lambda col: col.map(train_id_order)
).reset_index(drop=True)

train_audio_dataset = RawAudioDataset(train_df_audio, train_audio_root, audio_label2id, name_map=fusion_name_map)

print(f"Train aligned: {len(train_texts_aligned)} samples")
print(f"Test aligned:  {len(texts_aligned)} samples")


Train aligned: 3350 samples
Test aligned:  2610 samples


In [21]:
import gc

# =====================
# Helper Functions to Get Probabilities
# =====================
def get_text_probs(model, tokenizer, texts, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for text in texts:
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

def get_image_probs(model, loader, device):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for imgs, _ in loader:
            b, t, c, h, w = imgs.shape
            imgs = imgs.view(b * t, c, h, w).to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            probs = probs.view(b, t, -1).mean(dim=1)
            probs_list.append(probs.cpu())
    return torch.cat(probs_list, dim=0)

def get_audio_probs_gpu(model, dataset, remap, device, max_seconds=5, batch_size=8):
    model.eval()
    model.to(device)
    max_samples = max_seconds * 16000
    probs_list = []
    with torch.no_grad():
        for i in range(0, len(dataset), batch_size):
            batch = [dataset[j] for j in range(i, min(i + batch_size, len(dataset)))]
            waveforms, _ = zip(*batch)
            waveforms = [w[:max_samples] for w in waveforms]
            inputs = processor(list(waveforms), sampling_rate=16000, return_tensors="pt", padding=True)
            outputs = model(inputs.input_values.to(device))
            probs = torch.softmax(outputs, dim=1)
            reordered = torch.zeros((probs.size(0), len(label_list)), device=device)
            for audio_idx, fusion_idx in remap.items():
                reordered[:, fusion_idx] = probs[:, audio_idx]
            probs_list.append(reordered.cpu())
    return torch.cat(probs_list, dim=0)

text_probs_train = get_text_probs(text_model, tokenizer, train_texts_aligned, device)
text_probs       = get_text_probs(text_model, tokenizer, texts_aligned, device)
text_model.cpu(); gc.collect(); torch.cuda.empty_cache()

image_probs_train = get_image_probs(image_model, train_image_loader, device)
image_probs       = get_image_probs(image_model, image_loader, device)
image_model.cpu(); gc.collect(); torch.cuda.empty_cache()

audio_probs_train = get_audio_probs_gpu(audio_model, train_audio_dataset, audio_to_fusion_idx, device)
audio_probs       = get_audio_probs_gpu(audio_model, audio_dataset, audio_to_fusion_idx, device)
audio_model.cpu(); gc.collect(); torch.cuda.empty_cache()

In [22]:
# =====================
# Single Modality Baselines
# =====================
fusion_labels = np.array(aligned_labels)

for name, probs in [("Text only", text_probs), ("Image only", image_probs), ("Audio only", audio_probs)]:
    preds = probs.numpy().argmax(axis=1)
    print(f"=== {name} ===")
    print("Accuracy:", accuracy_score(fusion_labels, preds))
    print("Weighted F1:", f1_score(fusion_labels, preds, average="weighted"))
    print()


=== Text only ===
Accuracy: 0.5789272030651341
Weighted F1: 0.5910542130272249

=== Image only ===
Accuracy: 0.2670498084291188
Weighted F1: 0.2597245245377185

=== Audio only ===
Accuracy: 0.16436781609195403
Weighted F1: 0.1874389057145349



In [23]:
# =====================
# Fusion 1: Image + Audio Only
# =====================
train_labels_arr = np.array(train_aligned_labels)
test_labels_arr  = np.array(aligned_labels)

train_ia = np.concatenate([image_probs_train.numpy(), audio_probs_train.numpy()], axis=1)
test_ia  = np.concatenate([image_probs.numpy(),       audio_probs.numpy()],       axis=1)

clf_ia = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=1000)
clf_ia.fit(train_ia, train_labels_arr)

test_preds_ia = clf_ia.predict(test_ia)
print("=== Image + Audio Fusion ===")
print("Accuracy:  ", accuracy_score(test_labels_arr, test_preds_ia))
print("Weighted F1: ", f1_score(test_labels_arr, test_preds_ia, average="weighted"))
print("\nClassification Report:\n", classification_report(test_labels_arr, test_preds_ia, target_names=label_list, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(test_labels_arr, test_preds_ia))


=== Image + Audio Fusion ===
Accuracy:   0.4781609195402299
Weighted F1:  0.32479433787708417

Classification Report:
               precision    recall  f1-score   support

       anger       0.31      0.06      0.09       345
     disgust       0.00      0.00      0.00        68
        fear       0.00      0.00      0.00        50
         joy       0.08      0.00      0.00       402
     neutral       0.48      0.98      0.65      1256
     sadness       0.00      0.00      0.00       208
    surprise       0.00      0.00      0.00       281

    accuracy                           0.48      2610
   macro avg       0.12      0.15      0.11      2610
weighted avg       0.29      0.48      0.32      2610


Confusion Matrix:
 [[  19    0    0    2  324    0    0]
 [   0    0    0    0   68    0    0]
 [   1    0    0    0   49    0    0]
 [  11    0    0    1  390    0    0]
 [  22    0    0    6 1228    0    0]
 [   1    0    0    1  206    0    0]
 [   7    0    0    3  271    0    0

c:\Users\User\Downloads\cds\venv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [24]:
# =====================
# Fusion 2: Image + Audio + Text
# =====================
train_iat = np.concatenate([text_probs_train.numpy(), image_probs_train.numpy(), audio_probs_train.numpy()], axis=1)
test_iat  = np.concatenate([text_probs.numpy(),       image_probs.numpy(),       audio_probs.numpy()],       axis=1)

fusion_clf = LogisticRegression(multi_class="multinomial", solver="lbfgs", max_iter=1000)
fusion_clf.fit(train_iat, train_labels_arr)

test_preds = fusion_clf.predict(test_iat)
print("=== Image + Audio + Text Fusion ===")
print("Accuracy:    ", accuracy_score(test_labels_arr, test_preds))
print("Weighted F1: ", f1_score(test_labels_arr, test_preds, average="weighted"))
print("\nClassification Report:\n", classification_report(test_labels_arr, test_preds, target_names=label_list, zero_division=0))
print("\nConfusion Matrix:\n", confusion_matrix(test_labels_arr, test_preds))


c:\Users\User\Downloads\cds\venv\lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


=== Image + Audio + Text Fusion ===
Accuracy:     0.6130268199233716
Weighted F1:  0.6104316187563713

Classification Report:
               precision    recall  f1-score   support

       anger       0.48      0.44      0.46       345
     disgust       0.37      0.24      0.29        68
        fear       0.20      0.24      0.22        50
         joy       0.58      0.56      0.57       402
     neutral       0.75      0.75      0.75      1256
     sadness       0.35      0.31      0.33       208
    surprise       0.51      0.65      0.57       281

    accuracy                           0.61      2610
   macro avg       0.46      0.46      0.46      2610
weighted avg       0.61      0.61      0.61      2610


Confusion Matrix:
 [[153  11   9  32  77  17  46]
 [ 12  16   1   3  22   6   8]
 [  5   1  12   1  19   6   6]
 [ 33   3   4 226  86  15  35]
 [ 64   9  24  78 947  70  64]
 [ 26   2   8  21  72  64  15]
 [ 27   1   2  27  38   4 182]]
